In [1]:
# =========================================================
# NOTEBOOK: 02_data_cleaning.ipynb
# =========================================================

# =========================================================
# STEP 1 — IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# =========================================================
# STEP 2 — LOAD DATASETS
# =========================================================

customers_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\customers.csv')

orders_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\orders.csv')

order_items_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\order_items.csv')

products_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\products.csv')

sellers_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\sellers.csv')

payments_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\order_payments.csv')

reviews_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\order_reviews.csv')

geolocation_df = pd.read_csv(r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\raw\geolocation.csv')

print("All Datasets Loaded Successfully")

All Datasets Loaded Successfully


In [3]:
# =========================================================
# STEP 3 — DATE CONVERSION
# =========================================================

orders_date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in orders_date_cols:
    orders_df[col] = pd.to_datetime(
        orders_df[col],
        errors='coerce'
    )

order_items_df['shipping_limit_date'] = pd.to_datetime(
    order_items_df['shipping_limit_date'],
    errors='coerce'
)

reviews_df['review_creation_date'] = pd.to_datetime(
    reviews_df['review_creation_date'],
    errors='coerce'
)

reviews_df['review_answer_timestamp'] = pd.to_datetime(
    reviews_df['review_answer_timestamp'],
    errors='coerce'
)

print("Date Conversion Completed")

Date Conversion Completed


In [4]:
# =========================================================
# STEP 4 — CHECK MISSING VALUES
# =========================================================

datasets = {
    'customers_df': customers_df,
    'orders_df': orders_df,
    'order_items_df': order_items_df,
    'products_df': products_df,
    'sellers_df': sellers_df,
    'payments_df': payments_df,
    'reviews_df': reviews_df,
    'geolocation_df': geolocation_df
}

print("\n================ MISSING VALUES ================\n")

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * 50)

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No Missing Values")
    else:
        print(missing.sort_values(ascending=False))



================ MISSING VALUES ================


customers_df
--------------------------------------------------
No Missing Values

orders_df
--------------------------------------------------
order_delivered_carrier_date     66252
order_delivered_customer_date    66252
dtype: int64

order_items_df
--------------------------------------------------
No Missing Values

products_df
--------------------------------------------------
No Missing Values

sellers_df
--------------------------------------------------
No Missing Values

payments_df
--------------------------------------------------
No Missing Values

reviews_df
--------------------------------------------------
No Missing Values

geolocation_df
--------------------------------------------------
No Missing Values


In [5]:
# =========================================================
# STEP 5 — HANDLE MISSING VALUES
# =========================================================

# -----------------------------
# ORDERS DATASET
# -----------------------------

orders_df['order_approved_at'] = (
    orders_df['order_approved_at']
    .fillna(orders_df['order_purchase_timestamp'])
)

# Delivery-related missing values
orders_df['order_delivered_carrier_date'] = (
    orders_df['order_delivered_carrier_date']
    .fillna(method='ffill')
)

orders_df['order_delivered_customer_date'] = (
    orders_df['order_delivered_customer_date']
    .fillna(method='ffill')
)

print("Orders Missing Values Handled")


# -----------------------------
# REVIEWS DATASET
# -----------------------------

reviews_df['review_comment_title'] = (
    reviews_df['review_comment_title']
    .fillna('No Title')
)

reviews_df['review_comment_message'] = (
    reviews_df['review_comment_message']
    .fillna('No Review')
)

print("Review Missing Values Handled")

Orders Missing Values Handled
Review Missing Values Handled


In [6]:
# =========================================================
# STEP 6 — CHECK DUPLICATES
# =========================================================

print("\n================ DUPLICATES ================\n")

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    print(f"{name} --> Duplicate Rows: {duplicate_count}")


================ DUPLICATES ================

customers_df --> Duplicate Rows: 0
orders_df --> Duplicate Rows: 0
order_items_df --> Duplicate Rows: 0
products_df --> Duplicate Rows: 0
sellers_df --> Duplicate Rows: 0
payments_df --> Duplicate Rows: 0
reviews_df --> Duplicate Rows: 0
geolocation_df --> Duplicate Rows: 0


In [7]:
# =========================================================
# STEP 7 — REMOVE DUPLICATES
# =========================================================

customers_df.drop_duplicates(inplace=True)
orders_df.drop_duplicates(inplace=True)
order_items_df.drop_duplicates(inplace=True)
products_df.drop_duplicates(inplace=True)
sellers_df.drop_duplicates(inplace=True)
payments_df.drop_duplicates(inplace=True)
reviews_df.drop_duplicates(inplace=True)
geolocation_df.drop_duplicates(inplace=True)

print("\nDuplicates Removed Successfully")


Duplicates Removed Successfully


In [8]:
# =========================================================
# STEP 8 — DATA CONSISTENCY CHECKS
# =========================================================

print("\n================ DATA CONSISTENCY CHECKS ================\n")

# Negative price check
negative_prices = (
    order_items_df[order_items_df['price'] < 0]
)

print(f"Negative Prices Rows: {negative_prices.shape[0]}")

# Negative freight values
negative_freight = (
    order_items_df[order_items_df['freight_value'] < 0]
)

print(f"Negative Freight Rows: {negative_freight.shape[0]}")

# Invalid review scores
invalid_reviews = (
    reviews_df[
        ~reviews_df['review_score'].between(1, 5)
    ]
)

print(f"Invalid Review Scores Rows: {invalid_reviews.shape[0]}")


================ DATA CONSISTENCY CHECKS ================

Negative Prices Rows: 0
Negative Freight Rows: 0
Invalid Review Scores Rows: 0


In [24]:
# =========================================================
# STEP 9 — OUTLIER DETECTION
# =========================================================

print("\n================ OUTLIER DETECTION ================\n")

numerical_cols = [
    'price',
    'freight_value',
    'discount_rate'
]

for col in numerical_cols:

    Q1 = order_items_df[col].quantile(0.25)
    Q3 = order_items_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    outliers = order_items_df[
        (order_items_df[col] < lower_bound) |
        (order_items_df[col] > upper_bound)
    ]

    print(f"{col} --> Outliers: {outliers.shape[0]}")


================ OUTLIER DETECTION ================

price --> Outliers: 0
freight_value --> Outliers: 0
discount_rate --> Outliers: 0


In [25]:
# =========================================================
# STEP 10 — FEATURE ENGINEERING
# =========================================================

# Delivery Duration
orders_df['delivery_duration_days'] = (
    orders_df['order_delivered_customer_date']
    - orders_df['order_purchase_timestamp']
).dt.days

# Delivery Delay
orders_df['delivery_delay_days'] = (
    orders_df['order_delivered_customer_date']
    - orders_df['order_estimated_delivery_date']
).dt.days

# Approval Time
orders_df['approval_time_hours'] = (
    orders_df['order_approved_at']
    - orders_df['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Purchase Month
orders_df['purchase_month'] = (
    orders_df['order_purchase_timestamp'].dt.month
)

# Purchase Year
orders_df['purchase_year'] = (
    orders_df['order_purchase_timestamp'].dt.year
)

# Purchase Weekday
orders_df['purchase_weekday'] = (
    orders_df['order_purchase_timestamp'].dt.day_name()
)

print("Feature Engineering Completed")


Feature Engineering Completed


In [27]:
# =========================================================
# STEP 11 — CREATE MASTER DATASET
# =========================================================

master_df = orders_df.merge(
    customers_df,
    on='customer_id',
    how='left'
)

master_df = master_df.merge(
    order_items_df,
    on='order_id',
    how='left'
)

master_df = master_df.merge(
    products_df,
    on='product_id',
    how='left'
)

master_df = master_df.merge(
    sellers_df,
    on='seller_id',
    how='left'
)

master_df = master_df.merge(
    payments_df,
    on='order_id',
    how='left'
)

master_df = master_df.merge(
    reviews_df,
    on='order_id',
    how='left'
)

print("\nMaster Dataset Created Successfully")

print("\nMaster Dataset Shape:")
print(master_df.shape)



Master Dataset Created Successfully

Master Dataset Shape:
(2530433, 55)


In [28]:
# =========================================================
# STEP 12 — CHECK FINAL MISSING VALUES
# =========================================================

print("\n================ FINAL MISSING VALUES ================\n")

final_missing = (
    master_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

final_missing = final_missing[final_missing > 0]

print(final_missing)


================ FINAL MISSING VALUES ================

review_score                     167729
review_comment_title             167729
review_comment_message           167729
review_id                        167729
review_creation_date             167729
review_answer_timestamp          167729
order_delivered_customer_date         3
delivery_delay_days                   3
delivery_duration_days                3
order_delivered_carrier_date          3
dtype: int64


In [29]:
# =========================================================
# STEP 13 — BASIC CLEANED DATASET INFO
# =========================================================

print("\n================ CLEANED MASTER DATASET INFO ================\n")

print(master_df.info())




================ CLEANED MASTER DATASET INFO ================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530433 entries, 0 to 2530432
Data columns (total 55 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   order_id                       object        
 1   customer_id                    object        
 2   order_status                   object        
 3   order_purchase_timestamp       datetime64[ns]
 4   order_approved_at              datetime64[ns]
 5   order_delivered_carrier_date   datetime64[ns]
 6   order_delivered_customer_date  datetime64[ns]
 7   order_estimated_delivery_date  datetime64[ns]
 8   delivery_duration_days         float64       
 9   delivery_delay_days            float64       
 10  approval_time_hours            float64       
 11  purchase_month                 int32         
 12  purchase_year                  int32         
 13  purchase_weekday               object        
 14  cu

In [30]:
# =========================================================
# STEP 14 — SAVE CLEANED DATASET
# =========================================================

master_df.to_csv(
    r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\processed\master_dataset.csv',
    index=False
)

print("\nCleaned Master Dataset Saved Successfully")


Cleaned Master Dataset Saved Successfully


In [31]:

# =========================================================
# STEP 15 — CLEANING SUMMARY
# =========================================================

print("\n===================================================")
print("DATA CLEANING SUMMARY")
print("===================================================")

print("""
Tasks Completed:
✔ Missing Value Handling
✔ Duplicate Removal
✔ Invalid Data Removal
✔ Outlier Treatment
✔ Date Conversion
✔ Feature Engineering
✔ Master Dataset Creation
✔ Cleaned Dataset Export

Output File:
data/processed/master_dataset.csv
""")


DATA CLEANING SUMMARY

Tasks Completed:
✔ Missing Value Handling
✔ Duplicate Removal
✔ Invalid Data Removal
✔ Outlier Treatment
✔ Date Conversion
✔ Feature Engineering
✔ Master Dataset Creation
✔ Cleaned Dataset Export

Output File:
data/processed/master_dataset.csv



In [32]:
# =========================================================
# STEP 16 — NOTEBOOK COMPLETION
# =========================================================

print("\n===================================================")
print("02_data_cleaning.ipynb COMPLETED SUCCESSFULLY")
print("===================================================")


02_data_cleaning.ipynb COMPLETED SUCCESSFULLY
